In [1]:
import tarfile

with tarfile.open("ROIs2017_winter_s1.tar.gz", "r:gz") as tar:
    members = tar.getnames()

# Print first 20 entries to understand the structure
for m in members[:20]:
    print(m)

ROIs2017_winter_s1
ROIs2017_winter_s1/s1_102
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p316.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p538.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p100.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p101.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p102.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p103.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p104.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p105.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p106.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p107.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p108.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p109.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p110.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p111.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p112.tif
ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p113.tif
ROIs2017_winter_s1/s1_102/R

In [6]:
import tarfile
from collections import Counter

with tarfile.open("ROIs2017_winter_s1.tar.gz", "r:gz") as tar:
    members = tar.getnames()

regions = []
for name in members:
    parts = name.split("/")
    if len(parts) >= 2 and parts[1]:  # skip root folder entry
        regions.append(parts[1])

counts = Counter(regions)
print(f"Total regions: {len(counts)}\n")
for region, count in sorted(counts.items()):
    print(f"{region}: {count} files")

Total regions: 27

s1_102: 720 files
s1_104: 509 files
s1_107: 776 files
s1_108: 784 files
s1_112: 830 files
s1_116: 778 files
s1_130: 781 files
s1_135: 480 files
s1_146: 613 files
s1_21: 558 files
s1_22: 611 files
s1_25: 717 files
s1_42: 785 files
s1_47: 673 files
s1_49: 735 files
s1_55: 343 files
s1_59: 711 files
s1_61: 557 files
s1_62: 624 files
s1_63: 785 files
s1_64: 785 files
s1_68: 744 files
s1_75: 701 files
s1_8: 785 files
s1_81: 610 files
s1_84: 607 files
s1_94: 758 files


In [5]:
import rasterio
from rasterio.crs import CRS

with tarfile.open("ROIs2017_winter_s1.tar.gz", "r:gz") as tar:
    # Pick the first .tif from each region
    tif = tar.extractfile("ROIs2017_winter_s1/s1_102/ROIs2017_winter_s1_102_p100.tif")
    with rasterio.open(tif) as src:
        print(src.crs)       # coordinate system
        print(src.bounds)    # bounding box (lat/lon if CRS is WGS84)

EPSG:32630
BoundingBox(left=704990.9725789141, bottom=5809773.7261494165, right=707550.9725789141, top=5812333.7261494165)


In [9]:
import tarfile
import rasterio
import io
from pyproj import Transformer
import reverse_geocoder

regions = [
    "s1_102", "s1_104", "s1_107", "s1_108", "s1_112", "s1_116",
    "s1_130", "s1_135", "s1_146", "s1_21", "s1_22", "s1_25",
    "s1_42", "s1_47", "s1_49", "s1_55", "s1_59", "s1_61",
    "s1_62", "s1_63", "s1_64", "s1_68", "s1_75", "s1_8",
    "s1_81", "s1_84", "s1_94"
]

# country code -> continent
cc_to_continent = {
    "AF": "Africa", "AX": "Europe", "AL": "Europe", "DZ": "Africa",
    "AS": "Oceania", "AD": "Europe", "AO": "Africa", "AI": "North America",
    "AQ": "Antarctica", "AG": "North America", "AR": "South America",
    "AM": "Asia", "AW": "North America", "AU": "Oceania", "AT": "Europe",
    "AZ": "Asia", "BS": "North America", "BH": "Asia", "BD": "Asia",
    "BB": "North America", "BY": "Europe", "BE": "Europe", "BZ": "North America",
    "BJ": "Africa", "BM": "North America", "BT": "Asia", "BO": "South America",
    "BA": "Europe", "BW": "Africa", "BR": "South America", "BN": "Asia",
    "BG": "Europe", "BF": "Africa", "BI": "Africa", "CV": "Africa",
    "KH": "Asia", "CM": "Africa", "CA": "North America", "KY": "North America",
    "CF": "Africa", "TD": "Africa", "CL": "South America", "CN": "Asia",
    "CO": "South America", "KM": "Africa", "CG": "Africa", "CD": "Africa",
    "CR": "North America", "CI": "Africa", "HR": "Europe", "CU": "North America",
    "CY": "Asia", "CZ": "Europe", "DK": "Europe", "DJ": "Africa",
    "DM": "North America", "DO": "North America", "EC": "South America",
    "EG": "Africa", "SV": "North America", "GQ": "Africa", "ER": "Africa",
    "EE": "Europe", "SZ": "Africa", "ET": "Africa", "FJ": "Oceania",
    "FI": "Europe", "FR": "Europe", "GA": "Africa", "GM": "Africa",
    "GE": "Asia", "DE": "Europe", "GH": "Africa", "GR": "Europe",
    "GD": "North America", "GT": "North America", "GN": "Africa",
    "GW": "Africa", "GY": "South America", "HT": "North America",
    "HN": "North America", "HK": "Asia", "HU": "Europe", "IS": "Europe",
    "IN": "Asia", "ID": "Asia", "IR": "Asia", "IQ": "Asia", "IE": "Europe",
    "IL": "Asia", "IT": "Europe", "JM": "North America", "JP": "Asia",
    "JO": "Asia", "KZ": "Asia", "KE": "Africa", "KI": "Oceania",
    "KP": "Asia", "KR": "Asia", "KW": "Asia", "KG": "Asia", "LA": "Asia",
    "LV": "Europe", "LB": "Asia", "LS": "Africa", "LR": "Africa",
    "LY": "Africa", "LI": "Europe", "LT": "Europe", "LU": "Europe",
    "MG": "Africa", "MW": "Africa", "MY": "Asia", "MV": "Asia",
    "ML": "Africa", "MT": "Europe", "MH": "Oceania", "MR": "Africa",
    "MU": "Africa", "MX": "North America", "FM": "Oceania", "MD": "Europe",
    "MC": "Europe", "MN": "Asia", "ME": "Europe", "MA": "Africa",
    "MZ": "Africa", "MM": "Asia", "NA": "Africa", "NR": "Oceania",
    "NP": "Asia", "NL": "Europe", "NZ": "Oceania", "NI": "North America",
    "NE": "Africa", "NG": "Africa", "MK": "Europe", "NO": "Europe",
    "OM": "Asia", "PK": "Asia", "PW": "Oceania", "PA": "North America",
    "PG": "Oceania", "PY": "South America", "PE": "South America",
    "PH": "Asia", "PL": "Europe", "PT": "Europe", "QA": "Asia",
    "RO": "Europe", "RU": "Europe", "RW": "Africa", "KN": "North America",
    "LC": "North America", "VC": "North America", "WS": "Oceania",
    "SM": "Europe", "ST": "Africa", "SA": "Asia", "SN": "Africa",
    "RS": "Europe", "SC": "Africa", "SL": "Africa", "SG": "Asia",
    "SK": "Europe", "SI": "Europe", "SB": "Oceania", "SO": "Africa",
    "ZA": "Africa", "SS": "Africa", "ES": "Europe", "LK": "Asia",
    "SD": "Africa", "SR": "South America", "SE": "Europe", "CH": "Europe",
    "SY": "Asia", "TW": "Asia", "TJ": "Asia", "TZ": "Africa", "TH": "Asia",
    "TL": "Asia", "TG": "Africa", "TO": "Oceania", "TT": "North America",
    "TN": "Africa", "TR": "Asia", "TM": "Asia", "TV": "Oceania",
    "UG": "Africa", "UA": "Europe", "AE": "Asia", "GB": "Europe",
    "US": "North America", "UY": "South America", "UZ": "Asia",
    "VU": "Oceania", "VE": "South America", "VN": "Asia", "YE": "Asia",
    "ZM": "Africa", "ZW": "Africa"
}

results = []

with tarfile.open("ROIs2017_winter_s1.tar.gz", "r:gz") as tar:
    members = tar.getmembers()

    region_sample = {}
    for m in members:
        parts = m.name.split("/")
        if len(parts) == 3 and m.name.endswith(".tif"):
            region = parts[1]
            if region not in region_sample:
                region_sample[region] = m

    for region in regions:
        if region not in region_sample:
            print(f"{region}: not found")
            continue

        member = region_sample[region]
        f = tar.extractfile(member)
        data = io.BytesIO(f.read())

        with rasterio.open(data) as src:
            crs = src.crs.to_epsg()
            cx = (src.bounds.left + src.bounds.right) / 2
            cy = (src.bounds.top + src.bounds.bottom) / 2

        transformer = Transformer.from_crs(f"EPSG:{crs}", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(cx, cy)
        results.append((region, lat, lon))

coords = [(lat, lon) for _, lat, lon in results]
geo = reverse_geocoder.search(coords)

print(f"\n{'Region':<12} {'Continent':<20} {'Lat':>8} {'Lon':>8}")
print("-" * 52)
for (region, lat, lon), g in zip(results, geo):
    continent = cc_to_continent.get(g['cc'], "Unknown")
    print(f"{region:<12} {continent:<20} {lat:>8.2f} {lon:>8.2f}")


Region       Continent                 Lat      Lon
----------------------------------------------------
s1_102       Europe                  52.33     0.01
s1_104       North America           38.84   -90.06
s1_107       Asia                    37.53   111.21
s1_108       Asia                    41.12    29.14
s1_112       Europe                  51.05     5.66
s1_116       North America           40.31   -76.82
s1_130       Asia                    31.30    48.65
s1_135       North America           39.00  -104.94
s1_146       Europe                  41.93    12.83
s1_21        Africa                  -6.75    32.13
s1_22        Africa                   6.91    30.76
s1_25        Oceania                -20.66   130.09
s1_42        Asia                    19.48   110.02
s1_47        Africa                 -30.02    20.13
s1_49        South America           -7.93   -37.36
s1_55        South America          -37.08   -70.12
s1_59        North America           30.85   -94.84
s1_61     

In [1]:
import tarfile
import rasterio
import io
import os
import urllib.request
from pyproj import Transformer
import reverse_geocoder

# ── Configuration ────────────────────────────────────────────
OUTPUT_DIR = "data"
AMERICA_CONTINENTS = {"North America", "South America"}
CONTINENT_FOLDER = {
    "North America": "north_america",
    "South America": "south_america",
}

FILES = [
    # Terminadas:
    # ("ROIs1158_spring_s1.tar.gz",        "s1",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1158_spring_s1.tar.gz"),
    # ("ROIs1158_spring_s2.tar.gz",        "s2",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1158_spring_s2.tar.gz"),
    # ("ROIs1158_spring_s2_cloudy.tar.gz", "s2_cloudy", "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1158_spring_s2_cloudy.tar.gz"),
    # ("ROIs1868_summer_s1.tar.gz",        "s1",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1868_summer_s1.tar.gz"),
    # ("ROIs1868_summer_s2.tar.gz",        "s2",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1868_summer_s2.tar.gz"),
    # ("ROIs1868_summer_s2_cloudy.tar.gz", "s2_cloudy", "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1868_summer_s2_cloudy.tar.gz"),
    # Salteadas por pesadas:
    ("ROIs1970_fall_s1.tar.gz",          "s1",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1970_fall_s1.tar.gz"),
    ("ROIs1970_fall_s2.tar.gz",          "s2",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1970_fall_s2.tar.gz"),
    ("ROIs1970_fall_s2_cloudy.tar.gz",   "s2_cloudy", "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs1970_fall_s2_cloudy.tar.gz"),
    # ("ROIs2017_winter_s1.tar.gz",        "s1",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs2017_winter_s1.tar.gz"),
    # ("ROIs2017_winter_s2.tar.gz",        "s2",        "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs2017_winter_s2.tar.gz"),
    # ("ROIs2017_winter_s2_cloudy.tar.gz", "s2_cloudy", "https://dataserv.ub.tum.de/s/m1554803/download?path=%2F&files=ROIs2017_winter_s2_cloudy.tar.gz"),
]

CC_TO_CONTINENT = {
    "AF": "Africa", "AL": "Europe", "DZ": "Africa", "AD": "Europe", "AO": "Africa",
    "AG": "North America", "AR": "South America", "AM": "Asia", "AU": "Oceania",
    "AT": "Europe", "AZ": "Asia", "BS": "North America", "BH": "Asia", "BD": "Asia",
    "BB": "North America", "BY": "Europe", "BE": "Europe", "BZ": "North America",
    "BJ": "Africa", "BT": "Asia", "BO": "South America", "BA": "Europe", "BW": "Africa",
    "BR": "South America", "BN": "Asia", "BG": "Europe", "BF": "Africa", "BI": "Africa",
    "CV": "Africa", "KH": "Asia", "CM": "Africa", "CA": "North America", "CF": "Africa",
    "TD": "Africa", "CL": "South America", "CN": "Asia", "CO": "South America",
    "CG": "Africa", "CD": "Africa", "CR": "North America", "CI": "Africa", "HR": "Europe",
    "CU": "North America", "CY": "Asia", "CZ": "Europe", "DK": "Europe", "DJ": "Africa",
    "DM": "North America", "DO": "North America", "EC": "South America", "EG": "Africa",
    "SV": "North America", "GQ": "Africa", "ER": "Africa", "EE": "Europe", "ET": "Africa",
    "FJ": "Oceania", "FI": "Europe", "FR": "Europe", "GA": "Africa", "GM": "Africa",
    "GE": "Asia", "DE": "Europe", "GH": "Africa", "GR": "Europe", "GT": "North America",
    "GN": "Africa", "GW": "Africa", "GY": "South America", "HT": "North America",
    "HN": "North America", "HK": "Asia", "HU": "Europe", "IS": "Europe", "IN": "Asia",
    "ID": "Asia", "IR": "Asia", "IQ": "Asia", "IE": "Europe", "IL": "Asia", "IT": "Europe",
    "JM": "North America", "JP": "Asia", "JO": "Asia", "KZ": "Asia", "KE": "Africa",
    "KP": "Asia", "KR": "Asia", "KW": "Asia", "KG": "Asia", "LA": "Asia", "LV": "Europe",
    "LB": "Asia", "LS": "Africa", "LR": "Africa", "LY": "Africa", "LT": "Europe",
    "LU": "Europe", "MG": "Africa", "MW": "Africa", "MY": "Asia", "ML": "Africa",
    "MT": "Europe", "MR": "Africa", "MU": "Africa", "MX": "North America", "MD": "Europe",
    "MN": "Asia", "ME": "Europe", "MA": "Africa", "MZ": "Africa", "MM": "Asia",
    "NA": "Africa", "NP": "Asia", "NL": "Europe", "NZ": "Oceania", "NI": "North America",
    "NE": "Africa", "NG": "Africa", "MK": "Europe", "NO": "Europe", "OM": "Asia",
    "PK": "Asia", "PA": "North America", "PG": "Oceania", "PY": "South America",
    "PE": "South America", "PH": "Asia", "PL": "Europe", "PT": "Europe", "QA": "Asia",
    "RO": "Europe", "RU": "Europe", "RW": "Africa", "SA": "Asia", "SN": "Africa",
    "RS": "Europe", "SL": "Africa", "SG": "Asia", "SK": "Europe", "SI": "Europe",
    "SO": "Africa", "ZA": "Africa", "SS": "Africa", "ES": "Europe", "LK": "Asia",
    "SD": "Africa", "SR": "South America", "SE": "Europe", "CH": "Europe", "SY": "Asia",
    "TW": "Asia", "TJ": "Asia", "TZ": "Africa", "TH": "Asia", "TG": "Africa",
    "TT": "North America", "TN": "Africa", "TR": "Asia", "TM": "Asia", "UG": "Africa",
    "UA": "Europe", "AE": "Asia", "GB": "Europe", "US": "North America", "UY": "South America",
    "UZ": "Asia", "VE": "South America", "VN": "Asia", "YE": "Asia", "ZM": "Africa",
    "ZW": "Africa"
}

# ── Helpers ───────────────────────────────────────────────────
def classify_new_regions(tar, known_regions):
    """Classify any regions not yet seen. Updates known_regions in place."""
    region_sample = {}
    for m in tar.getmembers():
        parts = m.name.split("/")
        if len(parts) == 3 and m.name.endswith(".tif"):
            region = parts[1]
            if region not in known_regions and region not in region_sample:
                region_sample[region] = m

    if not region_sample:
        return

    print(f"  Classifying {len(region_sample)} new region(s)...")
    coords, names = [], []
    for region, member in region_sample.items():
        try:
            f = tar.extractfile(member)
            with rasterio.open(io.BytesIO(f.read())) as src:
                crs = src.crs.to_epsg()
                cx = (src.bounds.left + src.bounds.right) / 2
                cy = (src.bounds.top + src.bounds.bottom) / 2
            lon, lat = Transformer.from_crs(f"EPSG:{crs}", "EPSG:4326", always_xy=True).transform(cx, cy)
            coords.append((lat, lon))
            names.append(region)
        except Exception as e:
            print(f"  Warning: skipping {region}: {e}")

    for region, g in zip(names, reverse_geocoder.search(coords)):
        continent = CC_TO_CONTINENT.get(g['cc'], "Unknown")
        known_regions[region] = continent
        print(f"  {region} → {continent}")


def extract_to_flat(tar, modality, known_regions):
    """Write American .tif files directly into the 6 destination folders."""
    count = 0
    for member in tar.getmembers():
        if not member.name.endswith(".tif"):
            continue
        parts = member.name.split("/")
        if len(parts) != 3:
            continue

        region = parts[1]
        continent = known_regions.get(region)
        if continent not in AMERICA_CONTINENTS:
            continue

        # Build flat filename: e.g. ROIs1158_spring_s1_21_p100.tif
        original_name = parts[2]                        # e.g. ROIs1158_spring_s1_21_p100.tif
        continent_folder = CONTINENT_FOLDER[continent]  # north_america / south_america
        dest_folder = os.path.join(OUTPUT_DIR, f"{continent_folder}_{modality}")
        dest_path = os.path.join(dest_folder, original_name)

        with tar.extractfile(member) as src, open(dest_path, "wb") as dst:
            dst.write(src.read())
        count += 1

    return count


# ── Setup output folders ──────────────────────────────────────
# for continent in ["north_america", "south_america"]:
#     for modality in ["s1", "s2", "s2_cloudy"]:
#         os.makedirs(os.path.join(OUTPUT_DIR, f"{continent}_{modality}"), exist_ok=True)

# ── Main loop ─────────────────────────────────────────────────
known_regions = {}  # {region_name: continent}

for filename, modality, url in FILES:
    print(f"\n{'='*60}")
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")

    with tarfile.open(filename, "r:gz") as tar:
        classify_new_regions(tar, known_regions)    
        count = extract_to_flat(tar, modality, known_regions)
        print(f"  Saved {count} files.")

    os.remove(filename)
    print(f"  Deleted {filename}.")

print("\nAll done.")


Download complete.
  Classifying 58 new region(s)...
Loading formatted geocoded file...
  s1_33 → North America
  s1_1 → North America
  s1_100 → Asia
  s1_104 → Europe
  s1_105 → North America
  s1_107 → Europe
  s1_109 → Asia
  s1_11 → Africa
  s1_110 → North America
  s1_112 → Asia
  s1_114 → North America
  s1_116 → Asia
  s1_119 → Europe
  s1_120 → North America
  s1_122 → Oceania
  s1_125 → North America
  s1_128 → Europe
  s1_131 → North America
  s1_133 → Europe
  s1_134 → Asia
  s1_135 → Asia
  s1_136 → Europe
  s1_139 → Asia
  s1_14 → North America
  s1_141 → Europe
  s1_142 → North America
  s1_144 → Europe
  s1_147 → Asia
  s1_148 → North America
  s1_149 → Europe
  s1_19 → Africa
  s1_22 → North America
  s1_26 → Asia
  s1_27 → North America
  s1_28 → Africa
  s1_3 → Europe
  s1_30 → Europe
  s1_31 → Africa
  s1_35 → Africa
  s1_37 → Africa
  s1_39 → South America
  s1_4 → Africa
  s1_40 → Africa
  s1_41 → North America
  s1_42 → South America
  s1_57 → Asia
  s1_6 → Euro